# 04 - Physical and Spatial Context (Bronze to Silver)

Generates deterministic, synthetic airport hierarchy, spatial, asset, and twin-graph records from `bronze_demo_config`, then conforms them into Silver Delta tables. No live Azure Digital Twins or Azure Maps calls are made.

**Prerequisite:** attach `AirportOpsLakehouse` and run notebooks 01-03. Re-running overwrites every table.

In [ ]:
import random
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, DoubleType,
                               BooleanType, TimestampType, DateType)

config = spark.table('bronze_demo_config').first().asDict()
assert int(config['random_seed']) >= 0 and config['is_synthetic'] is True
BASE = datetime.strptime(config['base_date'], '%Y-%m-%d')
OBSERVATION_TS = config['observation_timestamp']
rng = random.Random(config['random_seed'] + 400)
BATCH_ID = f"PHYSICAL-{config['base_date'].replace('-', '')}-SEED-{config['random_seed']}"

airports = [row.asDict() for row in spark.table('bronze_airport').orderBy('airport_id').collect()]
gates = [row.asDict() for row in spark.table('bronze_gate').orderBy('gate_id').collect()]
assert len(airports) == int(config['airport_count'])
assert len(gates) == int(config['airport_count']) * int(config['gates_per_airport'])


def add_metadata(frame, classification):
    defaults = {
        'data_classification': F.lit(classification), 'source_name': F.lit('SyntheticSpatialGenerator'),
        'source_url': F.lit('repo://notebooks/04_Generate_Physical_Spatial_Context'),
        'source_as_of_date': F.lit(config['base_date']), 'generated_at_utc': F.current_timestamp(),
        'generator_version': F.lit(config['generator_version']), 'random_seed': F.lit(config['random_seed']),
        'batch_id': F.lit(BATCH_ID), 'record_source': F.lit('SyntheticSpatialGenerator')
    }
    for column_name, expression in defaults.items():
        if column_name not in frame.columns:
            frame = frame.withColumn(column_name, expression)
    return frame


def overwrite(rows, schema, table_name):
    frame = add_metadata(spark.createDataFrame(rows, schema=schema), 'SyntheticMaster')
    frame.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(table_name)
    print(table_name, frame.count())
    return frame


def twin(kind, entity_id):
    return kind.lower() + ':' + entity_id


def square_wkt(lon, lat, half_size):
    return ('POLYGON ((' + str(lon-half_size) + ' ' + str(lat-half_size) + ', ' +
            str(lon+half_size) + ' ' + str(lat-half_size) + ', ' +
            str(lon+half_size) + ' ' + str(lat+half_size) + ', ' +
            str(lon-half_size) + ' ' + str(lat+half_size) + ', ' +
            str(lon-half_size) + ' ' + str(lat-half_size) + '))')

In [ ]:
# Bronze: source-shaped physical and spatial registries.
airport_spatial_rows = []
terminal_zone_rows = []
checkpoint_rows = []
stand_rows = []
asset_rows = []

checkpoint_layout = [
    ('CheckIn', 'CHK', 'T1', 'Z1', -0.0012, -0.0004),
    ('Security', 'SEC', 'T1', 'Z2', 0.0012, 0.0004),
    ('Immigration', 'IMM', 'T2', 'Z1', -0.0012, -0.0004),
    ('Boarding', 'BRD', 'T2', 'Z2', 0.0012, 0.0004)
]
asset_types = [('Jetbridge', 'JTB'), ('BaggageBelt', 'BGB'), ('HVAC', 'HVC'),
               ('Escalator', 'ESC'), ('Lighting', 'LGT')]

for airport in airports:
    airport_id = airport['airport_id']
    base_lon = float(airport['longitude'])
    base_lat = float(airport['latitude'])
    airport_spatial_rows.append((
        airport_id, 'SyntheticSpatialGenerator', 'EPSG:4326', base_lon, base_lat,
        'POINT (' + str(base_lon) + ' ' + str(base_lat) + ')', twin('Airport', airport_id),
        'airport-' + airport_id.lower(), OBSERVATION_TS, True))

    for terminal_number in range(1, config['terminals_per_airport'] + 1):
        terminal_code = 'T' + str(terminal_number)
        terminal_id = airport_id + '-' + terminal_code
        terminal_lon = base_lon + (-0.003 if terminal_number == 1 else 0.003)
        terminal_lat = base_lat
        for zone_number in range(1, config['zones_per_terminal'] + 1):
            zone_code = 'Z' + str(zone_number)
            zone_id = terminal_id + '-' + zone_code
            zone_lon = terminal_lon
            zone_lat = terminal_lat + (-0.001 if zone_number == 1 else 0.001)
            zone_type = 'PublicProcessing' if zone_number == 1 else 'SecureOperations'
            terminal_zone_rows.append((
                terminal_id, airport_id, terminal_code, 'Terminal ' + terminal_code,
                terminal_number - 1, terminal_lon, terminal_lat, square_wkt(terminal_lon, terminal_lat, 0.0025),
                zone_id, zone_code, zone_type.replace('Processing', ' Processing').replace('Operations', ' Operations'),
                zone_type, zone_number - 1, zone_lon, zone_lat, square_wkt(zone_lon, zone_lat, 0.0009),
                twin('Terminal', terminal_id), twin('Zone', zone_id),
                'terminal-' + terminal_id.lower(), 'zone-' + zone_id.lower(), OBSERVATION_TS, True))

    for checkpoint_name, checkpoint_code, terminal_code, zone_code, dx, dy in checkpoint_layout:
        terminal_id = airport_id + '-' + terminal_code
        zone_id = terminal_id + '-' + zone_code
        terminal_lon = base_lon + (-0.003 if terminal_code == 'T1' else 0.003)
        checkpoint_id = airport_id + '-CP-' + checkpoint_code
        checkpoint_rows.append((
            checkpoint_id, airport_id, terminal_id, zone_id, checkpoint_name, checkpoint_name,
            terminal_lon + dx, base_lat + dy, twin('Checkpoint', checkpoint_id),
            'checkpoint-' + checkpoint_id.lower(), OBSERVATION_TS, True))

for gate in gates:
    airport = next(a for a in airports if a['airport_id'] == gate['airport_id'])
    airport_id = gate['airport_id']
    gate_id = gate['gate_id']
    gate_number = int(gate['gate_code'][1:])
    terminal_code = gate['terminal']
    terminal_id = airport_id + '-' + terminal_code
    zone_id = terminal_id + '-Z2'
    base_lon = float(airport['longitude']) + (-0.003 if terminal_code == 'T1' else 0.003)
    base_lat = float(airport['latitude']) + (gate_number - 3.5) * 0.00035
    stand_id = airport_id + '-S' + str(gate_number)
    stand_rows.append((stand_id, airport_id, terminal_id, gate_id, 'Stand ' + str(gate_number),
                       base_lon + 0.0015, base_lat, twin('Stand', stand_id),
                       'stand-' + stand_id.lower(), OBSERVATION_TS, True))
    for asset_type, asset_code in asset_types:
        asset_id = gate_id + '-AST-' + asset_code
        criticality = 'High' if asset_type in ['Jetbridge', 'BaggageBelt', 'HVAC'] else 'Medium'
        asset_rows.append((asset_id, airport_id, terminal_id, zone_id, gate_id, asset_type,
                           asset_type + ' at ' + gate_id, 'MaintenanceAsset', criticality,
                           twin('MaintenanceAsset', asset_id), 'asset-' + asset_id.lower(),
                           base_lon + 0.0002 * (asset_types.index((asset_type, asset_code)) + 1),
                           base_lat, OBSERVATION_TS, True))
    meter_id = gate_id + '-MTR-ENERGY'
    asset_rows.append((meter_id, airport_id, terminal_id, zone_id, gate_id, 'EnergyMeter',
                       'Energy meter at ' + gate_id, 'EnergyMeter', 'Medium',
                       twin('EnergyMeter', meter_id), 'asset-' + meter_id.lower(),
                       base_lon + 0.0012, base_lat, OBSERVATION_TS, True))

In [ ]:
# Bronze: deterministic Azure Digital Twins relationship graph export.
relationship_rows = []
def add_relationship(source_id, name, target_id):
    relationship_id = source_id.replace(':', '-') + '-' + name + '-' + target_id.replace(':', '-')
    relationship_rows.append((relationship_id, source_id, name, target_id, OBSERVATION_TS, True))

for row in terminal_zone_rows:
    terminal_id, airport_id, zone_id = row[0], row[1], row[8]
    if not any(r[1] == twin('Airport', airport_id) and r[3] == twin('Terminal', terminal_id) for r in relationship_rows):
        add_relationship(twin('Airport', airport_id), 'contains', twin('Terminal', terminal_id))
    add_relationship(twin('Terminal', terminal_id), 'contains', twin('Zone', zone_id))
for row in checkpoint_rows:
    add_relationship(twin('Zone', row[3]), 'contains', twin('Checkpoint', row[0]))
for gate in gates:
    gate_id = gate['gate_id']
    terminal_id = gate['airport_id'] + '-' + gate['terminal']
    stand_id = next(r[0] for r in stand_rows if r[3] == gate_id)
    add_relationship(twin('Terminal', terminal_id), 'contains', twin('Gate', gate_id))
    add_relationship(twin('Gate', gate_id), 'locatedIn', twin('Terminal', terminal_id))
    add_relationship(twin('Gate', gate_id), 'serves', twin('Stand', stand_id))
for asset in asset_rows:
    asset_id, zone_id, gate_id, asset_class = asset[0], asset[3], asset[4], asset[7]
    add_relationship(twin('Zone', zone_id), 'contains', twin(asset_class, asset_id))
    add_relationship(twin(asset_class, asset_id), 'locatedIn', twin('Zone', zone_id))
    if asset_class == 'MaintenanceAsset':
        meter_id = gate_id + '-MTR-ENERGY'
        add_relationship(twin('MaintenanceAsset', asset_id), 'monitoredBy', twin('EnergyMeter', meter_id))

In [ ]:
# Persist Bronze with explicit schemas.
airport_spatial_schema = 'airport_id string, source_system string, crs string, longitude double, latitude double, geometry_wkt string, twin_id string, map_feature_id string, source_updated_at timestamp, is_synthetic boolean'
terminal_zone_schema = 'terminal_id string, airport_id string, terminal_code string, terminal_name string, floor_count int, terminal_longitude double, terminal_latitude double, terminal_geometry_wkt string, zone_id string, zone_code string, zone_name string, zone_type string, floor_level int, zone_longitude double, zone_latitude double, zone_geometry_wkt string, terminal_twin_id string, zone_twin_id string, terminal_map_feature_id string, zone_map_feature_id string, source_updated_at timestamp, is_synthetic boolean'
checkpoint_schema = 'checkpoint_id string, airport_id string, terminal_id string, zone_id string, checkpoint_code string, checkpoint_name string, longitude double, latitude double, twin_id string, map_feature_id string, source_updated_at timestamp, is_synthetic boolean'
stand_schema = 'stand_id string, airport_id string, terminal_id string, gate_id string, stand_name string, longitude double, latitude double, twin_id string, map_feature_id string, source_updated_at timestamp, is_synthetic boolean'
asset_schema = 'asset_id string, airport_id string, terminal_id string, zone_id string, gate_id string, asset_type string, asset_name string, asset_class string, criticality string, twin_id string, map_feature_id string, longitude double, latitude double, source_updated_at timestamp, is_synthetic boolean'
relationship_schema = 'relationship_id string, source_twin_id string, relationship_name string, target_twin_id string, source_updated_at timestamp, is_synthetic boolean'

overwrite(airport_spatial_rows, airport_spatial_schema, 'bronze_airport_spatial')
overwrite(terminal_zone_rows, terminal_zone_schema, 'bronze_terminal_zones')
overwrite(checkpoint_rows, checkpoint_schema, 'bronze_checkpoint_registry')
overwrite(stand_rows, stand_schema, 'bronze_stand_registry')
overwrite(asset_rows, asset_schema, 'bronze_asset_registry')
overwrite(relationship_rows, relationship_schema, 'bronze_twin_relationships')

In [ ]:
# Silver dimensions: type, deduplicate, conform stable business keys, and retain lineage/quality metadata.
def save_silver(frame, table_name):
    frame = add_metadata(frame, 'DerivedAnalytical')
    if 'data_quality_status' not in frame.columns:
        frame = frame.withColumn('data_quality_status', F.lit('Valid'))
    if 'rejection_reason' not in frame.columns:
        frame = frame.withColumn('rejection_reason', F.lit(None).cast('string'))
    frame.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(table_name)
    print(table_name, frame.count())

terminal_source = spark.table('bronze_terminal_zones')
save_silver(terminal_source.select('terminal_id', 'airport_id', 'terminal_code', 'terminal_name',
                                   'floor_count', 'terminal_twin_id', 'terminal_map_feature_id')
            .dropDuplicates(['terminal_id']), 'dim_terminal')
save_silver(terminal_source.select('zone_id', 'terminal_id', 'airport_id', 'zone_code', 'zone_name',
                                   'zone_type', 'floor_level', 'zone_twin_id', 'zone_map_feature_id')
            .dropDuplicates(['zone_id']), 'dim_zone')
save_silver(spark.table('bronze_checkpoint_registry').dropDuplicates(['checkpoint_id']), 'dim_checkpoint')
save_silver(spark.table('bronze_stand_registry').dropDuplicates(['stand_id']), 'dim_stand')
save_silver(spark.table('bronze_asset_registry').dropDuplicates(['asset_id']), 'dim_asset')

date_schema = 'date_key int, calendar_date date, year int, month_number int, month_name string, day_of_month int'
simulation_days = max(1, (int(config['simulation_hours']) + 23) // 24)
date_rows = []
for day_offset in range(simulation_days):
    current_date = (BASE + timedelta(days=day_offset)).date()
    date_rows.append((int(current_date.strftime('%Y%m%d')), current_date, current_date.year,
                      current_date.month, current_date.strftime('%B'), current_date.day))
save_silver(spark.createDataFrame(date_rows, date_schema), 'dim_date')

time_rows = [(hour, f'{hour:02d}:00', 'Peak' if hour in [6,7,8,17,18,19] else 'OffPeak') for hour in range(24)]
save_silver(spark.createDataFrame(time_rows, 'hour int, hour_label string, operating_period string'), 'dim_time')

In [ ]:
# Silver location dimension and explicit many-to-many bridges.
location_schema = 'location_id string, location_type string, airport_id string, terminal_id string, zone_id string, longitude double, latitude double, spatial_ref string, twin_id string, is_synthetic boolean'
location_rows = []
for row in airport_spatial_rows:
    location_rows.append(('LOC-' + row[0], 'Airport', row[0], None, None, row[3], row[4], 'geojson:airports#' + row[7], row[6], True))
for row in terminal_zone_rows:
    if not any(x[0] == 'LOC-' + row[0] for x in location_rows):
        location_rows.append(('LOC-' + row[0], 'Terminal', row[1], row[0], None, row[5], row[6], 'geojson:terminals#' + row[18], row[16], True))
    location_rows.append(('LOC-' + row[8], 'Zone', row[1], row[0], row[8], row[13], row[14], 'geojson:zones#' + row[19], row[17], True))
for row in checkpoint_rows:
    location_rows.append(('LOC-' + row[0], 'Checkpoint', row[1], row[2], row[3], row[6], row[7], 'geojson:zones#zone-' + row[3].lower(), row[8], True))
for row in stand_rows:
    location_rows.append(('LOC-' + row[0], 'Stand', row[1], row[2], None, row[5], row[6], 'geojson:stands#' + row[8], row[7], True))
for row in asset_rows:
    location_rows.append(('LOC-' + row[0], 'Asset', row[1], row[2], row[3], row[11], row[12], 'geojson:gates#gate-' + row[4].lower(), row[9], True))
save_silver(spark.createDataFrame(location_rows, location_schema).dropDuplicates(['location_id']), 'dim_location')

asset_location_schema = 'asset_id string, location_id string, effective_from timestamp, effective_to timestamp, is_current boolean'
asset_location_rows = [(row[0], 'LOC-' + row[0], BASE, None, True) for row in asset_rows]
save_silver(spark.createDataFrame(asset_location_rows, asset_location_schema), 'bridge_asset_location')
gate_stand_schema = 'gate_id string, stand_id string, relationship_type string, effective_from timestamp, is_current boolean'
gate_stand_rows = [(row[3], row[0], 'AdjacentAndServedBy', BASE, True) for row in stand_rows]
save_silver(spark.createDataFrame(gate_stand_rows, gate_stand_schema), 'bridge_gate_stand')

In [ ]:
# Silver facts: deterministic asset state and queue-derived zone occupancy.
asset_state_schema = 'asset_state_id string, asset_id string, airport_id string, terminal_id string, zone_id string, event_time timestamp, health_status string, availability_pct double, anomaly_flag boolean, telemetry_age_min int, is_synthetic boolean'
asset_state_rows = []
for asset in asset_rows:
    for hour in range(0, config['simulation_hours'], config['asset_state_interval_hours']):
        deterministic_marker = sum(ord(ch) for ch in asset[0]) + hour + config['random_seed']
        anomaly = deterministic_marker % 19 == 0
        health_status = 'Anomalous' if anomaly else ('Degraded' if deterministic_marker % 11 == 0 else 'Healthy')
        availability = 82.0 + rng.randint(0, 5) if anomaly else (94.0 + rng.randint(0, 4) if health_status == 'Degraded' else 99.0)
        asset_state_rows.append(('AS-' + asset[0] + '-' + str(hour).zfill(2), asset[0], asset[1], asset[2], asset[3],
                                 BASE + timedelta(hours=hour), health_status, float(availability), anomaly,
                                 int((config['simulation_hours'] - hour - 1) * 60), True))
save_silver(spark.createDataFrame(asset_state_rows, asset_state_schema), 'fact_asset_state')

checkpoint_map = spark.table('dim_checkpoint').select('airport_id', F.col('checkpoint_code').alias('checkpoint'), 'checkpoint_id', 'terminal_id', 'zone_id')
zone_occupancy = (spark.table('fact_passenger_queue_metrics').alias('q')
    .join(checkpoint_map.alias('c'), ['airport_id', 'checkpoint'], 'inner')
    .select(F.concat(F.lit('ZO-'), F.col('queue_metric_id')).alias('zone_occupancy_id'),
            'airport_id', 'terminal_id', 'zone_id', 'checkpoint_id', 'event_time', 'event_hour',
            F.col('queue_length').cast('int').alias('occupancy_count'),
            F.col('throughput_pax').cast('int').alias('throughput_pax'),
            F.col('wait_time_min').cast('double').alias('wait_time_min'), F.lit(True).alias('is_synthetic')))
save_silver(zone_occupancy.dropDuplicates(['zone_occupancy_id']), 'fact_zone_occupancy')

In [ ]:
# Mandatory local integrity and configuration-derived volume checks.
def duplicate_count(table_name, key_columns):
    return spark.table(table_name).groupBy(*key_columns).count().filter(F.col('count') > 1).count()

for table_name, key in [('dim_terminal','terminal_id'),('dim_zone','zone_id'),('dim_checkpoint','checkpoint_id'),('dim_stand','stand_id'),('dim_asset','asset_id')]:
    assert duplicate_count(table_name, [key]) == 0
assert spark.table('dim_terminal').join(spark.table('dim_airport'), 'airport_id', 'left_anti').count() == 0
assert spark.table('dim_zone').join(spark.table('dim_terminal'), 'terminal_id', 'left_anti').count() == 0
assert spark.table('dim_checkpoint').join(spark.table('dim_zone'), 'zone_id', 'left_anti').count() == 0
assert spark.table('bridge_gate_stand').join(spark.table('dim_gate'), 'gate_id', 'left_anti').count() == 0
assert spark.table('bridge_gate_stand').join(spark.table('dim_stand'), 'stand_id', 'left_anti').count() == 0
assert spark.table('fact_asset_state').join(spark.table('dim_asset'), 'asset_id', 'left_anti').count() == 0
assert spark.table('fact_zone_occupancy').join(spark.table('dim_zone'), 'zone_id', 'left_anti').count() == 0

airport_count = int(config['airport_count'])
terminal_count = airport_count * int(config['terminals_per_airport'])
zone_count = terminal_count * int(config['zones_per_terminal'])
checkpoint_count = airport_count * int(config['checkpoints_per_airport'])
gate_count = airport_count * int(config['gates_per_airport'])
asset_count = gate_count * 6
asset_observations = asset_count * (int(config['simulation_hours']) // int(config['asset_state_interval_hours']))
queue_observations = airport_count * int(config['checkpoints_per_airport']) * (int(config['simulation_hours']) * 60 // int(config['queue_interval_minutes']))
relationship_count = terminal_count + zone_count + checkpoint_count + gate_count * 3 + asset_count * 2 + gate_count * 5

expected = {
    'dim_terminal': terminal_count, 'dim_zone': zone_count, 'dim_checkpoint': checkpoint_count,
    'dim_stand': gate_count, 'dim_asset': asset_count, 'fact_asset_state': asset_observations,
    'fact_zone_occupancy': queue_observations, 'bronze_twin_relationships': relationship_count
}
for table_name, expected_count in expected.items():
    actual_count = spark.table(table_name).count()
    assert actual_count == expected_count, f'{table_name}: {actual_count} != {expected_count}'
print('PASS: physical/spatial Bronze and Silver checks', expected)

In [ ]:
# Add gate locations so every DTDL/GeoJSON physical entity resolves through dim_location.
gate_location_rows = []
for gate in gates:
    airport = next(item for item in airports if item['airport_id'] == gate['airport_id'])
    gate_number = int(gate['gate_code'][1:])
    terminal_id = gate['airport_id'] + '-' + gate['terminal']
    zone_id = terminal_id + '-Z2'
    longitude = float(airport['longitude']) + (-0.003 if gate['terminal'] == 'T1' else 0.003)
    latitude = float(airport['latitude']) + (gate_number - 3.5) * 0.00035
    gate_location_rows.append((
        'LOC-' + gate['gate_id'], 'Gate', gate['airport_id'], terminal_id, zone_id,
        longitude, latitude, 'geojson:gates#gate-' + gate['gate_id'].lower(),
        twin('Gate', gate['gate_id']), True))
gate_locations = spark.createDataFrame(gate_location_rows, location_schema)
all_locations = spark.table('dim_location').unionByName(gate_locations, allowMissingColumns=True).dropDuplicates(['location_id'])
save_silver(all_locations, 'dim_location')
expected_location_count = (
    int(config['airport_count'])
    + int(config['airport_count']) * int(config['terminals_per_airport'])
    + int(config['airport_count']) * int(config['terminals_per_airport']) * int(config['zones_per_terminal'])
    + int(config['airport_count']) * int(config['checkpoints_per_airport'])
    + int(config['airport_count']) * int(config['gates_per_airport']) * 2
    + int(config['airport_count']) * int(config['gates_per_airport']) * 6
)
assert spark.table('dim_location').count() == expected_location_count
print('PASS: complete spatial join surface includes', len(gates), 'gates and', expected_location_count, 'locations')

In [ ]:
# Add date/hour keys required by the governed shared semantic model.
asset_state_with_time = (spark.table('fact_asset_state')
    .withColumn('date_key', F.date_format('event_time', 'yyyyMMdd').cast('int'))
    .withColumn('event_hour', F.hour('event_time')))
save_silver(asset_state_with_time, 'fact_asset_state')
zone_occupancy_with_date = (spark.table('fact_zone_occupancy')
    .withColumn('date_key', F.date_format('event_time', 'yyyyMMdd').cast('int')))
save_silver(zone_occupancy_with_date, 'fact_zone_occupancy')
assert {'date_key', 'event_hour'}.issubset(set(spark.table('fact_asset_state').columns))
assert 'date_key' in spark.table('fact_zone_occupancy').columns
print('PASS: physical observation facts include semantic date/hour keys')